In [1]:

import torch
from torch.utils.data import Dataset

class HyperspectralDataset(Dataset):

    def __init__(self, cube, labels, patch_size=3):
        self.cube = torch.tensor(
            cube, dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels, dtype=torch.long
        )

        self.patch_size = patch_size
        self.pad = patch_size // 2

        self.cube = torch.nn.functional.pad(
            self.cube.permute(2, 0, 1).unsqueeze(0),
            (self.pad, self.pad, self.pad, self.pad),
            mode="reflect"
        ).squeeze(0).permute(1, 2, 0)

    def __len__(self):
        return self.labels.shape[0] * self.labels.shape[1]

    def __getitem__(self, idx):

        height, width = self.labels.shape

        row = idx // width
        col = idx % width

        patch = self.cube[
            row:row + self.patch_size,
            col:col + self.patch_size,
            :
        ]

        patch = patch.permute(2, 0, 1)

        patch = patch.unsqueeze(0)

        label = self.labels[row, col]

        return patch, label


# Dataloaders

In [ ]:
from torch.utils.data import DataLoader, Subset
import numpy as np

dataset = HyperspectralDataset(cube, labels)

indices = np.arange(len(dataset))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels.reshape(-1)
)

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)